# 1. Langchain Agent

* OpenAI API Key 연결

In [11]:
# 운영체제 관련 기능(파일 경로, 환경변수 접근 등)을 사용하기 위한 표준 라이브러리
import os

# .env 파일에 저장된 환경변수를 불러오기 위한 함수
from dotenv import load_dotenv

# 경고(warning) 메시지를 제어하기 위한 표준 라이브러리
import warnings

# 실행 중 출력되는 경고 메시지를 무시하도록 설정
# 예: 라이브러리 버전 관련 경고, deprecated 경고 등이 화면에 안 나오게 됨
warnings.filterwarnings("ignore")

# 지정한 경로에 있는 .env 파일을 읽어서
# 그 안의 환경변수들을 현재 파이썬 실행 환경에 로드함
# 예를 들어 OPENAI_API_KEY=... 같은 값들을 코드에서 사용할 수 있게 됨
load_dotenv("/home/user/MultiAgents/.env")

True

### Tool을 활용하는 Agent 구현
* Agent는 Tool과 함께 구현됩니다.

In [12]:
# LangChain의 에이전트 생성 함수를 불러옵니다.
from langchain.agents import create_agent

# OpenAI 호환 채팅 모델 클래스를 불러옵니다.
from langchain_openai import ChatOpenAI

# 도시 이름을 받아 간단한 날씨 문자열을 반환하는 함수를 정의합니다.
def get_weather(city: str) -> str:
    """주어진 도시의 날씨를 제공합니다."""
    # 입력 문자열에 "서울"이 포함되면 서울 날씨를 반환합니다.
    if "서울" in city:
        return f"{city}의 날씨는 비가오고 있습니다."
    # 서울이 아니면 항상 맑다고 가정한 문자열을 반환합니다.
    return f"It's always sunny in {city}!"

# 프록시 환경변수를 사용해 ChatOpenAI 모델 객체를 생성합니다.
llm = ChatOpenAI(
    model="gpt-4o",
    api_key=os.environ["PROXY_TOKEN"],
    base_url=os.environ["PROXY_URL"],  # OpenAI API 대신 프록시 서버 주소를 사용합니다.
    temperature=0,
)

# 모델, 도구, 시스템 프롬프트를 묶어 에이전트를 생성합니다.
agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)

# 사용자 질문을 에이전트에 전달해 실행 결과를 받습니다.
res = agent.invoke(
    {"messages": [{"role": "user", "content": "서울의 날씨는 어때?"}]}
)

In [13]:
# res 안에 저장된 messages 리스트를 하나씩 꺼내며 반복합니다.
for msg in res['messages']:
    # 각 메시지를 사람이 보기 좋은 형식으로 출력합니다.
    msg.pretty_print()

================================ Human Message =================================

서울의 날씨는 어때?
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_VfMJZwKtmPKsejLAClmx8Moc)
 Call ID: call_VfMJZwKtmPKsejLAClmx8Moc
  Args:
    city: 서울
================================= Tool Message =================================
Name: get_weather

서울의 날씨는 비가오고 있습니다.
================================== Ai Message ==================================

서울의 날씨는 현재 비가 오고 있습니다.


## Build a real-world agent

1. 에이전트 제어를 위한 **시스템 프롬프트 작성**
2. 외부 데이터를 불러오는 **도구 제작**
3. 일관된 응답을 위한 **모델 구성**
4. 예측 결과를 활용하기 위한 **구조화된 출력**
5. 채팅을 위한 **대화 내용 기억**


### 시스템 프롬프트 정의
* Agent가 정확히 동작하도록 역할과 행동을 정의합니다.

In [14]:
SYSTEM_PROMPT = """당신은 언어유희(말장난)를 섞어 말하는 전문 기상 캐스터입니다.

당신은 다음 두 가지 도구에 접근할 수 있습니다:

- get_weather_for_location: 특정 지역의 날씨 정보를 가져올 때 사용하세요.
- get_user_location: 사용자의 위치 정보를 가져올 때 사용하세요.

사용자가 날씨를 물어볼 경우, 반드시 대상 위치를 확인해야 합니다.
만약 사용자의 질문 문맥상 현재 자신이 있는 곳의 날씨를 묻는 것이라고 판단된다면,
get_user_location 도구를 사용하여 사용자의 위치를 찾으세요."""

### 도구(tool) 제작
* 사용자가 정의한 `함수`를 호출해 모델이 외부 시스템과 상호 작용할 수 있도록 합니다.
* 도구는 Agent와 사용자의 대화 도중 호출이 결정되며, 에이전트 메모리와도 상호작용합니다.

```
도구(Tools)는 문서화가 잘 되어 있어야 합니다.
도구의 이름, 설명, 그리고 인자(argument) 이름들이 모델 프롬프트의 일부로 포함되기 때문입니다.

LangChain의 @tool 데코레이터는 메타데이터를 추가해 주며,
ToolRuntime 파라미터를 통해 런타임 주입(runtime injection)을 가능하게 합니다.
```

In [15]:
# 데이터 클래스를 만들기 위한 dataclass 데코레이터를 불러옵니다.
from dataclasses import dataclass

# LangChain의 tool 데코레이터와 ToolRuntime 타입을 불러옵니다.
from langchain.tools import tool, ToolRuntime

# 도시 이름을 입력받아 날씨 문자열을 반환하는 도구를 정의합니다.
@tool
def get_weather_for_location(city: str) -> str:
    """Get weather for a given city."""
    # 입력된 도시의 날씨를 간단한 문자열로 반환합니다.
    return f"It's always sunny in {city}!"

# 사용자 정의 런타임 컨텍스트 구조를 데이터 클래스로 정의합니다.
@dataclass
class Context:
    """Custom runtime context schema."""
    # 현재 사용자 식별자를 저장하는 필드입니다.
    user_id: str

# 런타임 컨텍스트에서 사용자 정보를 읽어 위치를 반환하는 도구를 정의합니다.
@tool
def get_user_location(runtime: ToolRuntime[Context]) -> str:
    """Retrieve user information based on user ID."""
    # 런타임 컨텍스트에서 현재 사용자 ID를 가져옵니다.
    user_id = runtime.context.user_id
    # 사용자 ID에 따라 서로 다른 위치 문자열을 반환합니다.
    return "Florida" if user_id == "1" else "SF"

### 모델 설정
* LLM을 설정합니다.

In [16]:
# LangChain에서 채팅 모델을 초기화하는 함수를 불러옵니다.
from langchain.chat_models import init_chat_model

# 프록시 환경변수를 사용해 gpt-4o-mini 채팅 모델을 초기화합니다.
model = init_chat_model(
    "gpt-4o-mini",
    api_key=os.environ["PROXY_TOKEN"],
    base_url=os.environ["PROXY_URL"],  # OpenAI API 대신 프록시 서버 주소를 사용합니다.
    temperature=0,
)

### 구조화된 출력
* 에이전트 응답이 특정 구조(형태)와 일치해야되는 경우, 구조화된 출력을 설정합니다.

In [17]:
# 데이터 클래스를 만들기 위한 dataclass 데코레이터를 불러옵니다.
from dataclasses import dataclass

# 에이전트의 구조화된 응답 형식을 정의하는 데이터 클래스를 선언합니다.
@dataclass
class ResponseFormat:
    """Response schema for the agent."""
    # 항상 포함되어야 하는 말장난 형식의 응답 필드입니다.
    punny_response: str
    # 날씨 관련 추가 정보가 있을 때만 채워지는 선택 필드입니다.
    weather_conditions: str | None = None

### Chatbot을 위한 메모리 추가
* Agent가 대화 내용을 기억하게 하기 위해 메모리를 추가해야합니다.
* 추후 DB등과 연결하여 사용할 수 있습니다.

In [18]:
# 메모리 기반 체크포인터 클래스를 불러옵니다.
from langgraph.checkpoint.memory import InMemorySaver

# 실행 중 상태를 메모리에 저장하는 체크포인터 객체를 생성합니다.
checkpointer = InMemorySaver()

### 최종 Agent 구성
* 설정된 내용들을 하나의 Agent로 통합해서 선언하고, 활용합니다.

In [19]:
# 구조화된 출력을 tool 호출 방식으로 처리하기 위한 ToolStrategy를 불러옵니다.
from langchain.agents.structured_output import ToolStrategy

# 모델, 도구, 컨텍스트, 출력 형식, 체크포인터를 포함한 에이전트를 생성합니다.
agent = create_agent(
    model=model,
    system_prompt=SYSTEM_PROMPT,
    tools=[get_user_location, get_weather_for_location],
    context_schema=Context,
    response_format=ToolStrategy(ResponseFormat),
    checkpointer=checkpointer
)

# thread_id를 이용해 같은 대화 세션을 구분하는 설정을 만듭니다.
config = {"configurable": {"thread_id": "1"}}

# 사용자 질문과 사용자 컨텍스트를 전달해 에이전트를 실행합니다.
response = agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather outside?"}]},
    config=config,
    context=Context(user_id="1")
)

# 구조화된 응답 결과를 출력합니다.
print(response['structured_response'])
# ResponseFormat(
#     punny_response="플로리다는 여전히 ‘해-맑은(sun-derful)’ 하루를 보내고 있어요! 햇빛이 종일 ‘ray-dio(라디오)’ 히트곡을 틀어주고 있네요! 지금이야말로 완벽한 ‘solar-bration(태양 축제)’을 즐길 날씨예요! 만약 비를 기대하셨다면 그 생각은 이미 ‘washed up(물거품)’이 되었어요 — 예보는 여전히 ‘clear-ly(아주 맑게)’ 훌륭합니다!",
#     weather_conditions="플로리다는 항상 맑아요!"
# ))

# 같은 thread_id로 감사 메시지를 보내 이전 대화를 이어서 실행합니다.
response = agent.invoke(
    {"messages": [{"role": "user", "content": "thank you!"}]},
    config=config,
    context=Context(user_id="1")
)

# 이어진 대화의 구조화된 응답 결과를 출력합니다.
print(response['structured_response'])
# ResponseFormat(
#     punny_response="언제든 ‘thund-erfully(천둥처럼 멋지게)’ 도와드릴게요! 날씨 소식을 ‘current(전류/최신)’하게 유지하도록 돕는 건 언제나 ‘breeze(산들바람/식은 죽 먹기)’죠. 저는 여기서 계속 ‘cloud(구름/기다리다)’처럼 머무르며 필요하실 때마다 예보를 ‘shower(퍼붓다)’해 드릴 준비가 되어 있어요. 플로리다의 햇살 아래 ‘sun-sational(태양처럼 멋진)’ 하루 보내세요!",
#     weather_conditions=None
# )

Deserializing unregistered type __main__.ResponseFormat from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'ResponseFormat')]


ResponseFormat(punny_response='Looks like Florida is serving up some sunshine on a silver platter!', weather_conditions="It's always sunny in Florida!")
ResponseFormat(punny_response="No problem! I'm always here to weather the storm of your questions!", weather_conditions=None)


### 전체코드 예시

In [20]:
from dataclasses import dataclass

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool, ToolRuntime
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.structured_output import ToolStrategy


# Define system prompt
SYSTEM_PROMPT = """You are an expert weather forecaster, who speaks in puns.

You have access to two tools:

- get_weather_for_location: use this to get the weather for a specific location
- get_user_location: use this to get the user's location

If a user asks you for the weather, make sure you know the location. If you can tell from the question that they mean wherever they are, use the get_user_location tool to find their location."""

# Define context schema
@dataclass
class Context:
    """Custom runtime context schema."""
    user_id: str

# Define tools
@tool
def get_weather_for_location(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

@tool
def get_user_location(runtime: ToolRuntime[Context]) -> str:
    """Retrieve user information based on user ID."""
    user_id = runtime.context.user_id
    return "Florida" if user_id == "1" else "SF"

# Configure model
model = init_chat_model(
    "gpt-4o-mini",
    api_key=os.environ["PROXY_TOKEN"],
    base_url=os.environ["PROXY_URL"],  # 핵심: OpenAI SDK의 base_url과 동일 개념
    temperature=0,    
)

# Define response format
@dataclass
class ResponseFormat:
    """Response schema for the agent."""
    # A punny response (always required)
    punny_response: str
    # Any interesting information about the weather if available
    weather_conditions: str | None = None

# Set up memory
checkpointer = InMemorySaver()

# Create agent
agent = create_agent(
    model=model,
    system_prompt=SYSTEM_PROMPT,
    tools=[get_user_location, get_weather_for_location],
    context_schema=Context,
    response_format=ToolStrategy(ResponseFormat),
    checkpointer=checkpointer
)

# Run agent
# `thread_id` is a unique identifier for a given conversation.
config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather outside?"}]},
    config=config,
    context=Context(user_id="1")
)

print(response['structured_response'])
# ResponseFormat(
#     punny_response="Florida is still having a 'sun-derful' day! The sunshine is playing 'ray-dio' hits all day long! I'd say it's the perfect weather for some 'solar-bration'! If you were hoping for rain, I'm afraid that idea is all 'washed up' - the forecast remains 'clear-ly' brilliant!",
#     weather_conditions="It's always sunny in Florida!"
# )


# Note that we can continue the conversation using the same `thread_id`.
response = agent.invoke(
    {"messages": [{"role": "user", "content": "thank you!"}]},
    config=config,
    context=Context(user_id="1")
)

print(response['structured_response'])
# ResponseFormat(
#     punny_response="You're 'thund-erfully' welcome! It's always a 'breeze' to help you stay 'current' with the weather. I'm just 'cloud'-ing around waiting to 'shower' you with more forecasts whenever you need them. Have a 'sun-sational' day in the Florida sunshine!",
#     weather_conditions=None
# )

Deserializing unregistered type __main__.ResponseFormat from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'ResponseFormat')]


ResponseFormat(punny_response="Looks like Florida is putting on its best show with sunny skies! It's a bright day to soak up some rays!", weather_conditions='sunny')
ResponseFormat(punny_response="You're welcome! I'm always here to forecast some fun! If you need more weather updates, just give me a shout!", weather_conditions=None)


## 연습문제

### 📝 문제 1: 기본 도구(Tool) 정의하기

**목표:** 두 수의 곱셈을 수행하는 함수를 만들고, 에이전트가 사용할 수 있는 `Tool`로 변환하세요.

**요구사항:**
1.  함수 이름: `multiply_numbers`
2.  입력: `a` (int), `b` (int)
3.  기능: 두 수의 곱 반환
4.  **필수:** `@tool` 데코레이터, 타입 힌트, Docstring(설명) 포함


In [21]:
from langchain.tools import tool

@tool
def multiply_numbers(a: int, b: int) -> int:
    """Multiply two integer numbers and return the result."""
    return a * b

# 확인용 코드
print(f"Tool 이름: {multiply_numbers.name}")
print(f"설명: {multiply_numbers.description}")

Tool 이름: multiply_numbers
설명: Multiply two integer numbers and return the result.


### 📝 문제 2: 런타임 컨텍스트(Context) 활용

**목표:** 실행 시점의 사용자 정보를 활용하여, 사용자의 **게임 레벨**을 알려주는 도구를 만드세요.

**요구사항:**
1.  `dataclass`를 사용하여 `GameContext` 정의 (속성: `level` (str))
2.  도구 이름: `check_user_level`
3.  기능: `ToolRuntime`을 통해 Context에 접근하여, "현재 레벨은 {level}입니다." 반환


In [22]:
from dataclasses import dataclass
from langchain.tools import tool, ToolRuntime

@dataclass
class GameContext:
    level: str

@tool
def check_user_level(runtime: ToolRuntime[GameContext]) -> str:
    """Checks the user's current game level from context."""
    level = runtime.context.level
    return f"현재 레벨은 {level}입니다."

# 확인용 코드
print(f"Tool 이름: {check_user_level.name}")
print(f"설명: {check_user_level.description}")

llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=os.environ["PROXY_TOKEN"],
    base_url=os.environ["PROXY_URL"],  # 핵심: OpenAI SDK의 base_url과 동일 개념
    temperature=0,
)

agent = create_agent(
    model=llm,
    tools=[check_user_level],
    context_schema=GameContext,
    system_prompt=(
        "사용자가 게임 레벨을 물어보면 반드시 check_user_level 도구를 사용해 "
        "현재 레벨을 확인하고 한국어로 답변하세요."
    ),
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "내 레벨 알려줘"}]},
    context=GameContext(level="100"),
)

# LangChain v1 에이전트는 보통 messages 기반으로 반환합니다.
# 구현/버전에 따라 result 형태가 약간 다를 수 있어 안전하게 출력합니다.
print("=== RAW RESULT ===")
print(result)

# 흔한 형태: result["messages"]의 마지막이 최종 답변
if isinstance(result, dict) and "messages" in result and result["messages"]:
    last_msg = result["messages"][-1]
    # last_msg가 dict일 수도, Message 객체일 수도 있어 방어적으로 처리
    content = last_msg.get("content") if isinstance(last_msg, dict) else getattr(last_msg, "content", None)
    print("\n=== FINAL ANSWER ===")
    print(content)

Tool 이름: check_user_level
설명: Checks the user's current game level from context.
=== RAW RESULT ===
{'messages': [HumanMessage(content='내 레벨 알려줘', additional_kwargs={}, response_metadata={}, id='3d0c1976-2bff-4138-b19c-5b1afde5a170'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 78, 'total_tokens': 90, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 29, 'engine_ttft_ms': 70, 'engine_ttlt_ms': 419, 'pre_inference_ms': 124, 'service_tbt_ms': 29, 'service_ttft_ms': 492, 'service_ttlt_ms': 836, 'total_duration_ms': 721, 'user_visible_ttft_ms': 368}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_4dcfea0a44', 'id': 'chatcmpl-Dk1B2kW3fkIriIhasXKxXW8n94APj', 'ser

### 📝 문제 3: 구조화된 출력(Structured Output)

**목표:** 에이전트가 분석 결과를 **요약(Summary)** 과 **감성(Sentiment)** 으로 나누어 답변하도록 설정하세요.

**요구사항:**
1.  `AnalysisResult` 데이터 클래스 정의 (`summary`: str, `sentiment`: str)
2.  `create_agent`를 사용할 때 `response_format`을 올바르게 설정

In [23]:
@dataclass
class AnalysisResult:
    """입력된 사용자의 리뷰를 요약한 후 감정분석을 진행해주세요."""
    summary: str
    sentiment: str

llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=os.environ["PROXY_TOKEN"],
    base_url=os.environ["PROXY_URL"],  # 핵심: OpenAI SDK의 base_url과 동일 개념
    temperature=0,
)

agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=(
        "사용자 리뷰를 분석해 핵심 내용을 한국어로 요약하고, "
        "감성을 긍정, 부정, 중립 중 하나로 분류하세요."
    ),
    response_format=ToolStrategy(AnalysisResult)
)

# 확인용 코드
# 테스트 입력: 긍정적인 리뷰 데이터
test_input = {"messages": [{"role": "user", "content": "이 제품은 정말 최고예요! 사용하기도 편하고 디자인도 예뻐서 강력 추천합니다."}]}

# 에이전트 실행
response = agent.invoke(test_input)

# 결과 출력
structured_res = response['structured_response']

print(f"결과 타입: {type(structured_res)}")  # <class '__main__.AnalysisResult'> 인지 확인
print(f"요약: {structured_res.summary}")
print(f"감성: {structured_res.sentiment}")

결과 타입: <class '__main__.AnalysisResult'>
요약: 이 제품은 사용하기 편하고 디자인이 예뻐서 추천한다.
감성: 긍정
